<a href="https://colab.research.google.com/github/meghana507/SCT_ML_4/blob/main/Handgesture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install mediapipe -q
!pip install scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 2.8 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp
import zipfile
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
from google.colab import files

In [3]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

print("Model file downloaded!")

Model file downloaded!


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_name = '/content/drive/MyDrive/SCT_TASK_4_Hand Gesture Recognition System.zip'

import zipfile
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

print("Extracted successfully!")

In [ ]:
# New API setup for version 0.10+
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create the landmarker with IMAGE mode (since we process images not video)
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

landmarker = HandLandmarker.create_from_options(options)

print("MediaPipe Hand Landmarker ready!")

In [ ]:
data = []    # stores 63 numbers per image (21 landmarks x 3 coordinates)
labels = []  # stores gesture name for each image
skipped = 0  # count of images where no hand was found

# Your dataset path inside colab
dataset_path = '/content/dataset/SCT_TASK_4_Hand Gesture Recognition System/leapGestRecog'

# If above path doesn't work run this to find correct path:
# import glob
# print(glob.glob('/content/dataset/**', recursive=True)[:20])

print("Starting landmark extraction...\n")

# Loop through subject folders (00, 01, 02...)
for subject in sorted(os.listdir(dataset_path)):
    subject_path = os.path.join(dataset_path, subject)

    if not os.path.isdir(subject_path):
        continue

    print(f"Processing subject: {subject}")

    # Loop through gesture folders (01_palm, 02_l, 03_fist...)
    for gesture_folder in sorted(os.listdir(subject_path)):
        gesture_path = os.path.join(subject_path, gesture_folder)

        if not os.path.isdir(gesture_path):
            continue

        gesture_name = gesture_folder  # use folder name as label

        # Loop through each image
        for img_file in os.listdir(gesture_path):
            img_path = os.path.join(gesture_path, img_file)

            # Read image using OpenCV
            img = cv2.imread(img_path)
            if img is None:
                continue

            # Convert BGR to RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Wrap image in MediaPipe format
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=img_rgb
            )

            # Run detection
            result = landmarker.detect(mp_image)

            # If hand found extract landmarks
            if result.hand_landmarks:
                hand = result.hand_landmarks[0]  # first hand

                # Each landmark has x, y, z → 21 x 3 = 63 values
                landmark_values = []
                for lm in hand:
                    landmark_values.append(lm.x)
                    landmark_values.append(lm.y)
                    landmark_values.append(lm.z)

                data.append(landmark_values)
                labels.append(gesture_name)
            else:
                skipped += 1

print(f"\nDone!")
print(f"Images with hand detected : {len(data)}")
print(f"Images skipped (no hand)  : {skipped}")
print(f"Gesture classes found     : {sorted(set(labels))}")

In [ ]:
# Convert lists to numpy arrays
X = np.array(data)
y = np.array(labels)

# Convert gesture names to numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Gesture classes and their codes:")
for i, name in enumerate(le.classes_):
    print(f"  {i} → {name}")

# Split data — 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTraining samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

# Train Random Forest
# n_estimators=100 means 100 decision trees voting together
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("\nModel trained successfully!")

In [ ]:
def predict_gesture(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = landmarker.detect(mp_image)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Extract landmarks
        landmark_values = []
        for lm in hand:
            landmark_values.append(lm.x)
            landmark_values.append(lm.y)
            landmark_values.append(lm.z)

        # Predict gesture
        prediction = model.predict([landmark_values])[0]
        gesture_name = le.inverse_transform([prediction])[0]
        confidence = max(model.predict_proba([landmark_values])[0]) * 100

        # Draw landmarks on image manually
        h, w, _ = img.shape
        for lm in hand:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(img, (cx, cy), 5, (0, 255, 0), -1)

        # Show result
        plt.figure(figsize=(6, 5))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"Gesture: {gesture_name}  |  Confidence: {confidence:.1f}%", fontsize=13)
        plt.axis('off')
        plt.show()

        print(f"Predicted Gesture : {gesture_name}")
        print(f"Confidence        : {confidence:.1f}%")

    else:
        print("No hand detected in this image")

# Change this path to any image from your dataset
predict_gesture('/content/dataset/SCT_TASK_4_Hand Gesture Recognition System/leapGestRecog/00/01_palm/frame_00_01_0001.png')

In [ ]:
with open('gesture_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Saved!")
files.download('gesture_model.pkl')
files.download('label_encoder.pkl')